# 🏭 Multicriteria Decision Analysis — Industtech Maroc

**Course**: Aide à la Décision Multicritère (ADM)  
**Author**: KANOHA ELENGA Helmie Naella Jihane  
**Method**: Analytic Hierarchy Process (AHP) + Linear Optimization  
**Reference**: Culaste et al. (ISAHP 2022) · Palma et al. (ISAHP 2024)

---

## 🎯 Problem Statement

Industtech Maroc, a fictional Moroccan industrial company, needs to select the most relevant **digital transformation projects** under a limited budget of **1 500 kMAD**.

This notebook documents **all AHP computations step by step**, from pairwise comparison matrices to final project rankings and optimal selection.

### Workflow
```
Step 1 — Define projects & criteria
Step 2 — Build criteria comparison matrix (AHP)
Step 3 — Compute criteria weights + consistency check
Step 4 — Build project comparison matrices (one per criterion)
Step 5 — Compute local scores per criterion
Step 6 — Compute global AHP scores
Step 7 — Linear optimization (budget constraint)
Step 8 — Export results to CSV
```

---
## 📦 Step 0 — Imports

In [ ]:
import numpy as np
import pandas as pd
import itertools
import os

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries loaded ✅')

---
## 📋 Step 1 — Projects & Criteria

### 1.1 Candidate Projects

In [ ]:
projects = {
    'id':          ['P1', 'P2', 'P3', 'P4', 'P5'],
    'name':        ['AutoProd', 'IoT-Predict', 'DataPerf', 'GreenEnergy', 'DigitalTwin'],
    'description': [
        'Production line automation (robots, conveyors)',
        'IoT sensors for predictive maintenance',
        'Performance analytics dashboard (ERP/MES)',
        'Energy efficiency solutions (solar + heat recovery)',
        'Digital twin of a manufacturing line'
    ],
    'cost_kmad':   [850, 420, 280, 560, 690]
}

df_projects = pd.DataFrame(projects)
print(f'Total budget needed if all selected: {sum(projects["cost_kmad"])} kMAD')
print(f'Available budget: 1500 kMAD\n')
df_projects

### 1.2 Evaluation Criteria

In [ ]:
criteria = {
    'id':          ['C1', 'C2', 'C3', 'C4', 'C5'],
    'name':        ['Impact économique', 'Faisabilité technique',
                    'Durabilité', 'Alignement stratégique', 'Complexité'],
    'description': [
        'ROI, productivity gains, cost reduction',
        'Technological maturity, available skills',
        'CO2 reduction, energy efficiency',
        'Alignment with Industry 4.0 vision',
        'Implementation risks, delays, dependencies (to minimize)'
    ],
    'type':        ['max', 'max', 'max', 'max', 'min']
}

df_criteria = pd.DataFrame(criteria)
df_criteria

---
## ⚖️ Step 2 — Saaty's Pairwise Comparison Scale

| Value | Interpretation |
|-------|----------------|
| 1     | Equal importance |
| 3     | Slightly more important |
| 5     | Noticeably more important |
| 7     | Very strongly more important |
| 9     | Absolutely more important |
| 2,4,6,8 | Intermediate values |
| 1/n   | Inverse (less important) |

### 2.1 Criteria Comparison Matrix

Judgments collected from the decision maker (pairwise):

In [ ]:
# Criteria pairwise comparison matrix
# Order: C1 (Eco), C2 (Tech), C3 (Sust), C4 (Strat), C5 (Complexity)
#
# Reading: M[i,j] = 'how much more important is criterion i vs criterion j?'
# M[j,i] = 1/M[i,j]  (reciprocal property)

M_criteria = np.array([
    # C1    C2    C3    C4    C5
    [1,    3,    1,    1/3,  5  ],  # C1 vs all
    [1/3,  1,    1/3,  1/3,  5  ],  # C2 vs all
    [1,    3,    1,    1/3,  5  ],  # C3 vs all
    [3,    3,    3,    1,    7  ],  # C4 vs all  ← strategic alignment is dominant
    [1/5,  1/5,  1/5,  1/7,  1  ],  # C5 vs all  ← complexity has low weight
])

crit_labels = ['C1 Eco.', 'C2 Tech.', 'C3 Sust.', 'C4 Strat.', 'C5 Compl.']
df_M_crit = pd.DataFrame(M_criteria, index=crit_labels, columns=crit_labels)
print('Criteria pairwise comparison matrix:')
df_M_crit

---
## 🔢 Step 3 — AHP Weight Computation

### 3.1 Helper Function

In [ ]:
def ahp_compute(M, labels=None):
    """
    Full AHP computation from a pairwise comparison matrix M.
    Returns: weights, lambda_max, CI, CR
    
    Steps:
      1. Normalize each column (divide by column sum)
      2. Average each row → priority vector (weights)
      3. Compute weighted sum vector
      4. Compute lambda_max (average of ratio weighted_sum / weights)
      5. Compute Consistency Index CI = (lambda_max - n) / (n - 1)
      6. Compute Consistency Ratio CR = CI / RI
    """
    n = M.shape[0]
    RI_table = {1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12,
                6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}

    # Step 1: Column sums
    col_sums = M.sum(axis=0)

    # Step 2: Normalized matrix
    M_norm = M / col_sums

    # Step 3: Priority vector (weights)
    weights = M_norm.mean(axis=1)

    # Step 4: Weighted sum vector
    weighted_sum = M @ weights

    # Step 5: Lambda max
    lambda_vec = weighted_sum / weights
    lambda_max = lambda_vec.mean()

    # Step 6: Consistency
    CI = (lambda_max - n) / (n - 1)
    RI = RI_table.get(n, 1.12)
    CR = CI / RI if RI > 0 else 0

    return {
        'weights': weights,
        'col_sums': col_sums,
        'M_norm': M_norm,
        'weighted_sum': weighted_sum,
        'lambda_max': lambda_max,
        'CI': CI,
        'CR': CR,
        'n': n
    }

print('Function ahp_compute() defined ✅')

### 3.2 Apply to Criteria Matrix

In [ ]:
res_crit = ahp_compute(M_criteria)

# Step-by-step display
print('── STEP 1: Column sums ──')
print(dict(zip(crit_labels, res_crit['col_sums'].round(4))))

print('\n── STEP 2: Normalized matrix ──')
df_norm = pd.DataFrame(res_crit['M_norm'], index=crit_labels, columns=crit_labels)
display(df_norm.round(4))

print('── STEP 3: Priority vector (weights) ──')
for label, w in zip(crit_labels, res_crit['weights']):
    bar = '█' * int(w * 50)
    print(f'  {label:<15} {w:.4f}  ({w*100:.1f}%)  {bar}')

print(f'\n── STEP 4: λ_max = {res_crit["lambda_max"]:.4f} ──')
print(f'── STEP 5: CI    = {res_crit["CI"]:.4f} ──')
print(f'── STEP 6: CR    = {res_crit["CR"]:.4f}  ', end='')
print('✅ CONSISTENT (< 0.10)' if res_crit['CR'] < 0.10 else '⚠️  INCONSISTENT (> 0.10)')

---
## 📊 Step 4 — Project Comparison Matrices

For each criterion, we compare the 5 projects pairwise.  
Judgments for **C4 (Strategic Alignment)** were provided by the decision maker.  
Judgments for C1, C2, C3, C5 are simulated based on domain knowledge.

### Project labels

In [ ]:
proj_labels = ['P1 AutoProd', 'P2 IoT', 'P3 DataPerf', 'P4 GreenEnergy', 'P5 DigitalTwin']

### 4.1 C1 — Economic Impact

In [ ]:
# AutoProd & DigitalTwin: high ROI potential (automation, twinning)
# IoT-Predict: medium-high (reduces maintenance costs)
# GreenEnergy: long-term savings
# DataPerf: moderate economic impact

M_C1 = np.array([
    [1,   3,   5,   3,   1  ],
    [1/3, 1,   3,   2,   1/3],
    [1/5, 1/3, 1,   1/2, 1/5],
    [1/3, 1/2, 2,   1,   1/3],
    [1,   3,   5,   3,   1  ]
])

res_C1 = ahp_compute(M_C1)
print(f'C1 — Economic Impact  |  CR = {res_C1["CR"]:.4f}',
      '✅' if res_C1['CR'] < 0.10 else '⚠️')
for p, w in zip(proj_labels, res_C1['weights']):
    print(f'  {p:<18} {w:.4f}')

### 4.2 C2 — Technical Feasibility

In [ ]:
# DataPerf: easiest to implement (standard BI tools)
# IoT-Predict & GreenEnergy: accessible technology
# AutoProd & DigitalTwin: complex, specialized skills required

M_C2 = np.array([
    [1,   1/3, 1/5, 1/3, 2  ],
    [3,   1,   1/2, 1,   3  ],
    [5,   2,   1,   3,   5  ],
    [3,   1,   1/3, 1,   3  ],
    [1/2, 1/3, 1/5, 1/3, 1  ]
])

res_C2 = ahp_compute(M_C2)
print(f'C2 — Technical Feasibility  |  CR = {res_C2["CR"]:.4f}',
      '✅' if res_C2['CR'] < 0.10 else '⚠️')
for p, w in zip(proj_labels, res_C2['weights']):
    print(f'  {p:<18} {w:.4f}')

### 4.3 C3 — Sustainability

In [ ]:
# GreenEnergy: clear winner (direct energy savings, CO2 reduction)
# IoT-Predict & DigitalTwin: reduce waste via predictive approaches
# AutoProd: energy-neutral
# DataPerf: minimal environmental impact

M_C3 = np.array([
    [1,   1/3, 3,   1/7, 1/3],
    [3,   1,   5,   1/5, 1  ],
    [1/3, 1/5, 1,   1/7, 1/5],
    [7,   5,   7,   1,   5  ],
    [3,   1,   5,   1/5, 1  ]
])

res_C3 = ahp_compute(M_C3)
print(f'C3 — Sustainability  |  CR = {res_C3["CR"]:.4f}',
      '✅' if res_C3['CR'] < 0.10 else '⚠️')
for p, w in zip(proj_labels, res_C3['weights']):
    print(f'  {p:<18} {w:.4f}')

### 4.4 C4 — Strategic Alignment *(decision maker judgments)*

In [ ]:
# Judgments provided by the decision maker during pairwise elicitation session:
# P1>P2=5, P1>P3=3, P1>P4=5, P1>P5=3
# P2>P3=1/5, P2>P4=3, P2>P5=1/3
# P3>P4=1, P3>P5=5
# P4>P5=1/5

M_C4 = np.array([
    [1,   5,   3,   5,   3  ],
    [1/5, 1,   1/5, 3,   1/3],
    [1/3, 5,   1,   1,   5  ],
    [1/5, 1/3, 1,   1,   1/5],
    [1/3, 3,   1/5, 5,   1  ]
])

res_C4 = ahp_compute(M_C4)
print(f'C4 — Strategic Alignment  |  CR = {res_C4["CR"]:.4f}',
      '✅' if res_C4['CR'] < 0.10 else '⚠️  Note: slight inconsistency, acceptable for academic context')
for p, w in zip(proj_labels, res_C4['weights']):
    print(f'  {p:<18} {w:.4f}')

### 4.5 C5 — Implementation Complexity *(lower = better)*

In [ ]:
# Note: for this criterion, score represents EASE (inverse of complexity)
# DataPerf: simplest (standard tools, no hardware)
# IoT-Predict: fairly simple (off-the-shelf sensors)
# GreenEnergy: intermediate
# AutoProd & DigitalTwin: most complex

M_C5 = np.array([
    [1,   1/5, 1/7, 1/3, 2  ],
    [5,   1,   1/3, 3,   5  ],
    [7,   3,   1,   5,   7  ],
    [3,   1/3, 1/5, 1,   3  ],
    [1/2, 1/5, 1/7, 1/3, 1  ]
])

res_C5 = ahp_compute(M_C5)
print(f'C5 — Complexity (ease)  |  CR = {res_C5["CR"]:.4f}',
      '✅' if res_C5['CR'] < 0.10 else '⚠️')
for p, w in zip(proj_labels, res_C5['weights']):
    print(f'  {p:<18} {w:.4f}')

---
## 🧮 Step 5 — Local Scores Summary

In [ ]:
LOCAL_SCORES = np.column_stack([
    res_C1['weights'],
    res_C2['weights'],
    res_C3['weights'],
    res_C4['weights'],
    res_C5['weights'],
])

crit_short = ['C1 Eco.', 'C2 Tech.', 'C3 Sust.', 'C4 Strat.', 'C5 Compl.']
df_local = pd.DataFrame(LOCAL_SCORES, index=projects['name'], columns=crit_short)
print('Local AHP scores (rows=projects, columns=criteria):')
df_local

---
## 🏆 Step 6 — Global AHP Scores

Global score = weighted sum of local scores:

$$S_i = \sum_{j=1}^{5} w_j \cdot s_{ij}$$

where $w_j$ = criteria weight and $s_{ij}$ = local score of project $i$ on criterion $j$.

In [ ]:
CRITERIA_WEIGHTS = res_crit['weights']

print('Criteria weights used:')
for label, w in zip(crit_short, CRITERIA_WEIGHTS):
    print(f'  {label:<15} {w:.4f}  ({w*100:.1f}%)')

GLOBAL_SCORES = LOCAL_SCORES @ CRITERIA_WEIGHTS

print('\nGlobal AHP scores:')
ranking = np.argsort(GLOBAL_SCORES)[::-1]
df_global = pd.DataFrame({
    'Project':      [projects['name'][i] for i in ranking],
    'Global Score': [GLOBAL_SCORES[i] for i in ranking],
    'Rank':         range(1, 6)
})
df_global

---
## 💰 Step 7 — Linear Optimization

**Objective**: maximize total AHP score of selected projects  
**Constraint**: total cost ≤ budget  
**Variables**: binary ($x_i \in \{0,1\}$) — 1 if project $i$ is selected

$$\max \sum_{i=1}^{5} S_i \cdot x_i \quad \text{s.t.} \quad \sum_{i=1}^{5} c_i \cdot x_i \leq B$$

In [ ]:
BUDGET = 1500
costs  = projects['cost_kmad']
scores = GLOBAL_SCORES.tolist()
names  = projects['name']

# Exhaustive combinatorial search (2^5 - 1 = 31 combinations)
all_results = []
for r in range(1, 6):
    for combo in itertools.combinations(range(5), r):
        total_cost  = sum(costs[i]  for i in combo)
        total_score = sum(scores[i] for i in combo)
        if total_cost <= BUDGET:
            all_results.append({
                'combination': combo,
                'projects':    ' + '.join([names[i] for i in combo]),
                'total_score': round(total_score, 4),
                'total_cost':  total_cost,
                'n_projects':  len(combo)
            })

df_results = pd.DataFrame(all_results).sort_values('total_score', ascending=False)
print(f'Total feasible combinations: {len(df_results)} / 31')
print(f'\nTop 5 combinations:')
df_results[['projects','total_score','total_cost','n_projects']].head()

In [ ]:
# Optimal solution
optimal = df_results.iloc[0]
opt_idx  = list(optimal['combination'])

print('=' * 55)
print('  ✅ OPTIMAL SOLUTION')
print('=' * 55)
print(f'  Budget: {BUDGET} kMAD')
print()
for i in opt_idx:
    print(f'  ✔  {names[i]:<18}  score={scores[i]:.4f}  cost={costs[i]} kMAD')
not_sel = [i for i in range(5) if i not in opt_idx]
for i in not_sel:
    print(f'  ✗  {names[i]:<18}  score={scores[i]:.4f}  cost={costs[i]} kMAD')
print()
print(f'  Total score : {optimal["total_score"]:.4f}')
print(f'  Total cost  : {optimal["total_cost"]} kMAD ({optimal["total_cost"]/BUDGET*100:.1f}% of budget)')
print(f'  Remaining   : {BUDGET - optimal["total_cost"]} kMAD')

---
## 💾 Step 8 — Export Results to CSV

In [ ]:
os.makedirs('../data', exist_ok=True)

# ── 1. Projects + costs + global scores + selection ──────────
df_export = pd.DataFrame({
    'id':           projects['id'],
    'name':         projects['name'],
    'description':  projects['description'],
    'cost_kmad':    projects['cost_kmad'],
    'score_C1':     LOCAL_SCORES[:, 0].round(4),
    'score_C2':     LOCAL_SCORES[:, 1].round(4),
    'score_C3':     LOCAL_SCORES[:, 2].round(4),
    'score_C4':     LOCAL_SCORES[:, 3].round(4),
    'score_C5':     LOCAL_SCORES[:, 4].round(4),
    'global_score': GLOBAL_SCORES.round(4),
    'selected':     [1 if i in opt_idx else 0 for i in range(5)]
})
df_export.to_csv('../data/donnees_projets.csv', index=False)
print('✅ donnees_projets.csv exported')

# ── 2. Criteria weights ───────────────────────────────────────
df_weights = pd.DataFrame({
    'id':     criteria['id'],
    'name':   criteria['name'],
    'weight': CRITERIA_WEIGHTS.round(4),
    'CR':     [res_crit['CR']] * 5
})
df_weights.to_csv('../data/criteres_poids.csv', index=False)
print('✅ criteres_poids.csv exported')

print('\nProjects data:')
df_export

---
## 📝 Summary

| Step | Output |
|------|--------|
| Criteria weights (AHP) | C4 Strat. **43%** · C1 Eco. 20.6% · C3 Sust. 20.6% · C2 Tech. 11.8% · C5 Compl. 4% |
| Consistency (criteria) | CR = 0.062 ✅ |
| AHP ranking | #1 AutoProd · #2 DataPerf · #3 GreenEnergy · #4 DigitalTwin · #5 IoT-Predict |
| Optimal selection | **IoT-Predict + DataPerf + GreenEnergy** (score=0.539, cost=1260 kMAD) |

**Key insight**: AutoProd ranks #1 individually but is too costly (850 kMAD) to allow for a high-scoring combination. The optimal portfolio selects 3 complementary projects that together maximize strategic value within budget.

---
*Exported files: `data/donnees_projets.csv` · `data/criteres_poids.csv`*